# EDA i metody wizualizacji: wpływ AI i social mediów na zdrowie i wyniki studentów

**Autor:** _uzupełnij_
**Zbiór danych:** [AI & Social Media Impact: Student Health & Grades (Kaggle)](https://www.kaggle.com/datasets/srisyra02/ai-and-social-media-impact-student-health-and-grades)

Notatnik jest podzielony zgodnie z treścią zadania:
- **Część I:** opis danych, statystyki opisowe, wykresy jednej zmiennej
- **Część II:** redukcja wymiarów (PCA, MDS, t-SNE) i obserwacje odstające

## 0. Przygotowanie środowiska i wczytanie danych

Dane **nie są zapisywane lokalnie**. Przy każdym uruchomieniu notatnika pobieramy je z Kaggle.
Adres URL jest zapisany w pliku `database` w folderze projektu.

In [52]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Sprawdzenie, czy notatnik działa na Pythonie z .venv
print(sys.executable)

c:\Users\xBrav\Documents\AAA Studia Magisterskie\Semestr II\Eksploracja danych\ai_social_media_data_analyzis\.venv\Scripts\python.exe


In [53]:
# Plik `database` to zwykły plik tekstowy z adresem strony Kaggle, więc czytamy go jak tekst (to nie jest moduł Pythona).
PAGE_URL = Path("database").read_text(encoding="utf-8").strip()

# Adres strony prowadzi do HTML-a, a nie do danych. Wycinamy z niego "właściciel/nazwa-zbioru"
# i składamy adres API Kaggle, który zwraca sam plik.
slug = PAGE_URL.split("/datasets/")[1].strip("/")
DATA_URL = f"https://www.kaggle.com/api/v1/datasets/download/{slug}"
print(DATA_URL)

# Kaggle zwraca archiwum ZIP, ale adres nie kończy się na .zip, więc podajemy kompresję jawnie.
df = pd.read_csv(DATA_URL, compression="zip")

https://www.kaggle.com/api/v1/datasets/download/srisyra02/ai-and-social-media-impact-student-health-and-grades


In [54]:
# Szybka kontrola, czy dane się wczytały
print(df.shape)   # (liczba wierszy, liczba kolumn)
df.head()

(16000, 10)


,Student_ID,Age,Gender,Education_Level,Daily_Social_Media_Hours,Daily_AI_Tool_Usage_Hours,Sleep_Hours,Physical_Activity_Hours,Mental_Health_Score,Physical_Health_Score
0,STU_00001,19,Male,College,5.32,1.21,7.30,1.41,74.06,98.90
1,STU_00002,16,Non-binary,High School,5.21,3.89,7.81,2.48,76.13,89.94
2,STU_00003,25,Male,University,3.61,2.65,6.34,0.06,65.01,76.75
3,STU_00004,23,Male,University,2.93,1.43,6.12,1.37,74.67,87.38
4,STU_00005,20,Non-binary,College,6.52,1.64,5.23,1.14,67.74,88.04


---
# Część I: opis, statystyka opisowa, wykresy zmiennych

## 1.1 Pochodzenie danych

Zbiór **AI & Social Media Impact: Student Health & Grades** pochodzi z platformy
[Kaggle](https://www.kaggle.com/datasets/srisyra02/ai-and-social-media-impact-student-health-and-grades).
Autorka: Srimathy Sivanessan. Wersja 1 z 5.09.2026, licencja CC0 (domena publiczna), plik `AI_SocialMedia_Student_Dataset.csv`.
Zbiór ma opisywać, jak korzystanie z AI i mediów społecznościowych wiąże się ze snem, aktywnością
i zdrowiem uczniów i studentów.

In [55]:
n_przypadkow, n_zmiennych = df.shape
liczbowe = df.select_dtypes(include="number").columns.tolist()
print(f"Przypadki: {n_przypadkow}, zmienne: {n_zmiennych} "
      f"(liczbowe: {len(liczbowe)}, tekstowe: {n_zmiennych - len(liczbowe)}, w tym identyfikator Student_ID)")

# Szybkie testy "czy dane są syntetyczne"
print("Braki danych:", df.isna().sum().sum(), "| duplikaty:", df.duplicated().sum())
print("Liczebność roczników (min–max):", df["Age"].value_counts().min(), "–", df["Age"].value_counts().max())
print("Przedziały wieku w każdym poziomie edukacji:")
print(df.groupby("Education_Level")["Age"].agg(["min", "max"]))
print("Odsetek osób z aktywnością fizyczną równą dokładnie 0:", f"{(df['Physical_Activity_Hours'] == 0).mean():.1%}")
print("Maks. |korelacja| wieku z innymi zmiennymi:", df[liczbowe].corr()["Age"].drop("Age").abs().max().round(3))

Przypadki: 16000, zmienne: 10 (liczbowe: 7, tekstowe: 3, w tym identyfikator Student_ID)
Braki danych: 0 | duplikaty: 0
Liczebność roczników (min–max): 1150 – 1312
Przedziały wieku w każdym poziomie edukacji:
                 min  max
Education_Level          
College           18   21
High School       13   17
University        22   25
Odsetek osób z aktywnością fizyczną równą dokładnie 0: 17.5%
Maks. |korelacja| wieku z innymi zmiennymi: 0.013


**Podsumowanie:** 16 000 przypadków, 10 zmiennych: 7 liczbowych, 2 jakościowe i identyfikator.
Dane są **niemal na pewno syntetyczne**. Nie ma braków ani duplikatów, każdy rocznik ma prawie tyle samo osób,
poziom edukacji wynika wprost z wieku bez wyjątków, 17,5% osób ma równo 0 h aktywności
(to ślad uciętego rozkładu normalnego), a wiek nie koreluje z niczym. Opis na Kaggle wspomina też o zmiennych,
których w pliku nie ma (wypalenie, wyniki w nauce).
Zbiór nadaje się do ćwiczenia metod, ale wniosków nie należy przenosić na prawdziwych studentów.

## 1.2 Tabela opisu zmiennych

In [56]:
opisy = {
    "Student_ID":                ("identyfikator studenta",                       "identyfikator"),
    "Age":                       ("wiek w latach (13–25)",                        "ilościowa dyskretna"),
    "Gender":                    ("płeć",                                         "jakościowa nominalna"),
    "Education_Level":           ("poziom edukacji",                              "jakościowa porządkowa"),
    "Daily_Social_Media_Hours":  ("dzienny czas w mediach społecznościowych [h]", "ilościowa ciągła"),
    "Daily_AI_Tool_Usage_Hours": ("dzienny czas korzystania z narzędzi AI [h]",   "ilościowa ciągła"),
    "Sleep_Hours":               ("dzienny czas snu [h]",                         "ilościowa ciągła"),
    "Physical_Activity_Hours":   ("dzienny czas aktywności fizycznej [h]",        "ilościowa ciągła"),
    "Mental_Health_Score":       ("ocena zdrowia psychicznego (0–100)",           "ilościowa ciągła"),
    "Physical_Health_Score":     ("ocena zdrowia fizycznego (0–100)",             "ilościowa ciągła"),
}

tabela_zmiennych = pd.DataFrame({
    "Zmienna":   df.columns,
    "Opis":      [opisy[k][0] for k in df.columns],
    "Typ":       [opisy[k][1] for k in df.columns],
    "Braki [%]": (df.isna().mean() * 100).round(2).values,
})
# Wyświetlenie z tekstem wyrównanym do lewej (Styler formatuje tabelę jak w HTML/CSS)
(tabela_zmiennych.style
    .hide(axis="index")
    .set_properties(**{"text-align": "left"})
    .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}])
    .format({"Braki [%]": "{:.2f}"}))

Zmienna,Opis,Typ,Braki [%]
Student_ID,identyfikator studenta,identyfikator,0.00
Age,wiek w latach (13–25),ilościowa dyskretna,0.00
Gender,płeć,jakościowa nominalna,0.00
Education_Level,poziom edukacji,jakościowa porządkowa,0.00
Daily_Social_Media_Hours,dzienny czas w mediach społecznościowych [h],ilościowa ciągła,0.00
Daily_AI_Tool_Usage_Hours,dzienny czas korzystania z narzędzi AI [h],ilościowa ciągła,0.00
Sleep_Hours,dzienny czas snu [h],ilościowa ciągła,0.00
Physical_Activity_Hours,dzienny czas aktywności fizycznej [h],ilościowa ciągła,0.00
Mental_Health_Score,ocena zdrowia psychicznego (0–100),ilościowa ciągła,0.00
Physical_Health_Score,ocena zdrowia fizycznego (0–100),ilościowa ciągła,0.00


## 1.3 Statystyki opisowe zmiennych ilościowych

In [57]:
ilosciowe = df.select_dtypes(include="number").columns   # 7 zmiennych liczbowych (Student_ID jest tekstem, więc odpada sam)

# Kolumny jak na tablicy: n, średnia (x̄), mediana (Me), odchylenie (s), Q1, Q3 oraz zakres jako min i max
statystyki = pd.DataFrame({
    "n":  df[ilosciowe].count(),
    "x̄":  df[ilosciowe].mean(),
    "Me": df[ilosciowe].median(),
    "s":  df[ilosciowe].std(),
    "Q1": df[ilosciowe].quantile(0.25),
    "Q3": df[ilosciowe].quantile(0.75),
    "min": df[ilosciowe].min(),
    "max": df[ilosciowe].max(),
})
statystyki.index.name = "Zmienna"
statystyki.round(2)

,n,x̄,Me,s,Q1,Q3,min,max
Zmienna,,,,,,,,
Age,16000,19.04,19.00,3.77,16.00,22.00,13.00,25.00
Daily_Social_Media_Hours,16000,4.54,4.50,2.40,2.84,6.18,0.00,14.00
Daily_AI_Tool_Usage_Hours,16000,2.59,2.52,1.68,1.31,3.75,0.00,9.50
Sleep_Hours,16000,6.55,6.54,1.27,5.68,7.41,2.00,11.15
Physical_Activity_Hours,16000,1.25,1.13,1.04,0.31,1.96,0.00,5.00
Mental_Health_Score,16000,72.49,73.75,9.24,66.76,79.55,32.56,91.76
Physical_Health_Score,16000,88.02,89.46,9.69,81.52,97.17,48.03,99.98


## 1.4 Zmienne jakościowe: liczność i procent

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

TLO, JASNY = "#16191e", "#e6e6e6"
jakosciowe = ["Gender", "Education_Level"]
KOLORY_KATEGORII = ["#4c8bd6", "#f0883e", "#5cc27a"]   # stała kolejność kolorów kategorii

fig = make_subplots(rows=len(jakosciowe), cols=2, column_widths=[0.45, 0.55],
                    specs=[[{"type": "table"}, {"type": "domain"}]] * len(jakosciowe),
                    subplot_titles=[t for kol in jakosciowe for t in (f"<b>{kol}</b>: tabela", "udział procentowy")],
                    vertical_spacing=0.12)

for i, kol in enumerate(jakosciowe, start=1):
    n = df[kol].value_counts()
    if kol == "Education_Level":   # zmienna porządkowa, więc kategorie w naturalnej kolejności
        n = n.reindex(["High School", "College", "University"])
    p = (n / n.sum() * 100).round(2)

    fig.add_trace(go.Table(
        columnwidth=[1.6, 1, 1],
        header=dict(values=["<b>Kategoria</b>", "<b>n</b>", "<b>p [%]</b>"], align="left",
                    fill_color="#2b3038", line_color="#3a3f47", font=dict(size=12, color=JASNY)),
        cells=dict(values=[list(n.index) + ["<b>Razem</b>"],
                           list(n.values) + [f"<b>{n.sum()}</b>"],
                           [f"{x:.2f}" for x in p.values] + ["<b>100.00</b>"]],
                   align="left", height=28, font=dict(size=12, color=JASNY), line_color="#3a3f47",
                   fill_color=[[TLO] * len(n) + ["#22262d"]])),
        row=i, col=1)

    fig.add_trace(go.Pie(
        labels=n.index, values=n.values, sort=False,
        marker=dict(colors=KOLORY_KATEGORII[:len(n)], line=dict(color=TLO, width=2)),
        texttemplate="%{label}<br>%{percent:.1%}", textposition="outside",
        hovertemplate="%{label}<br>n = %{value}<br>%{percent:.2%}<extra></extra>",
        showlegend=False), row=i, col=2)

fig.update_layout(height=330 * len(jakosciowe), width=1000, template="plotly_dark",
                  paper_bgcolor=TLO, plot_bgcolor=TLO, font=dict(color=JASNY),
                  title=dict(text="Zmienne jakościowe: liczność (n) i procent (p)", x=0.5),
                  margin=dict(t=90, l=20, r=20, b=20))
fig.show()

**Podsumowanie 1.2–1.4:** żadna zmienna nie ma braków. Średnie i mediany są zbliżone, więc rozkłady
są mniej więcej symetryczne. Wyjątkiem są oceny zdrowia, gdzie średnia jest niższa od mediany, czyli rozkład
ma lewy ogon (kilka osób z bardzo niskim wynikiem). Zmienne mają bardzo różne skale: godziny to 0–14,
a oceny zdrowia to 30–100. Ma to duże znaczenie dla PCA na macierzy kowariancji w Części II.
Płeć: kobiet i mężczyzn jest po ok. 48%, osób niebinarnych ok. 4%. Najliczniejszy poziom edukacji to szkoła średnia (38%).

## 1.5 Histogramy i wykresy pudełkowe

Dla wszystkich 7 zmiennych ilościowych. Wykresy interaktywne (plotly): najechanie myszką pokazuje wartości. Wykres pudełkowy jak na zajęciach: pudełko od Q1 do Q3,
kwadrat w medianie (Me), wąsy od minimum do maksimum. Przy takich wąsach obserwacji odstających
nie widać na wykresie, dlatego pod spodem liczymy je regułą 1,5 × IQR.
Zmienne jakościowe opisuje tabela procentowa z punktu 1.4. Histogram ma sens tylko dla liczb.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Kolory pod ciemny motyw
TLO = "#16191e"
KOLOR = "#4c8bd6"
CZERWONY = "#ff6b6b"
JASNY = "#e6e6e6"
SIATKA, SIATKA_POM = "#3a3f47", "#262a30"

JEDNOSTKI = {"Age": "lata", "Daily_Social_Media_Hours": "h", "Daily_AI_Tool_Usage_Hours": "h",
             "Sleep_Hours": "h", "Physical_Activity_Hours": "h",
             "Mental_Health_Score": "pkt", "Physical_Health_Score": "pkt"}

fig = make_subplots(rows=len(ilosciowe), cols=2, column_widths=[0.7, 0.3],
                    horizontal_spacing=0.07, vertical_spacing=0.035,
                    subplot_titles=[t for kol in ilosciowe for t in (f"<b>{kol}</b>: histogram", "wykres pudełkowy")])

for i, kol in enumerate(ilosciowe, start=1):
    s, jedn = df[kol], JEDNOSTKI[kol]
    st = {"max": s.max(), "Q3": s.quantile(0.75), "Me": s.median(), "Q1": s.quantile(0.25), "min": s.min()}
    os_h, os_b = fig.get_subplot(i, 1), fig.get_subplot(i, 2)
    xh, yh = os_h.xaxis.plotly_name.replace("axis", ""), os_h.yaxis.plotly_name.replace("axis", "")
    xb, yb = os_b.xaxis.plotly_name.replace("axis", ""), os_b.yaxis.plotly_name.replace("axis", "")

    # ---------- Histogram (przedziały liczymy sami, żeby notatnik nie przechowywał 16 000 punktów)
    krawedzie = np.arange(s.min() - 0.5, s.max() + 1.5) if kol == "Age" else np.histogram_bin_edges(s, bins=40)
    licznosci, krawedzie = np.histogram(s, bins=krawedzie)
    fig.add_trace(go.Bar(
        x=(krawedzie[:-1] + krawedzie[1:]) / 2, y=licznosci, width=np.diff(krawedzie),
        marker_color=KOLOR, marker_line=dict(color=TLO, width=0.6),
        customdata=np.c_[krawedzie[:-1], krawedzie[1:]],
        hovertemplate=f"przedział: %{{customdata[0]:.2f}} – %{{customdata[1]:.2f}} {jedn}<br>liczność: %{{y}}<extra></extra>",
        showlegend=False), row=i, col=1)

    fig.add_vline(x=s.mean(), line=dict(color=CZERWONY, dash="dash", width=1.5), row=i, col=1)
    fig.add_vline(x=st["Me"], line=dict(color=JASNY, width=1.5), row=i, col=1)
    lewy_rog = s.skew() < 0   # legenda po stronie "lżejszego" ogona rozkładu
    fig.add_annotation(
        xref=f"{xh} domain", yref=f"{yh} domain", x=0.01 if lewy_rog else 0.99, y=0.98,
        xanchor="left" if lewy_rog else "right", yanchor="top", showarrow=False, align="left",
        text=(f"<span style='color:{CZERWONY}'>- - x̄ = {s.mean():.2f}</span><br>"
              f"—— Me = {st['Me']:.2f}<br>s = {s.std():.2f}"),
        bgcolor="rgba(22,25,30,0.9)", bordercolor="#555b66", borderwidth=1, font=dict(size=11, color=JASNY))
    fig.update_yaxes(range=[0, licznosci.max() * 1.3], title_text="liczność", title_font_size=10, row=i, col=1)
    fig.update_xaxes(title_text=f"{kol} [{jedn}]", title_font_size=10, row=i, col=1)

    # ---------- Wykres pudełkowy jak na tablicy: pudełko Q1–Q3, kwadrat w medianie, wąsy do min i max
    fig.add_trace(go.Box(
        x=[0], q1=[st["Q1"]], median=[st["Me"]], q3=[st["Q3"]],
        lowerfence=[st["min"]], upperfence=[st["max"]], width=0.5, name=kol,
        fillcolor="rgba(76,139,214,0.45)", line=dict(color=JASNY, width=1.2),
        showlegend=False), row=i, col=2)   # najechanie myszką pokazuje dokładne wartości
    fig.add_trace(go.Scatter(
        x=[0], y=[st["Me"]], mode="markers", hoverinfo="skip", showlegend=False,
        marker=dict(symbol="square", size=10, color=TLO, line=dict(color=JASNY, width=1.5))),
        row=i, col=2)
    # Legenda z wartościami obok pudełka (zamiast podpisów przy każdym elemencie)
    fig.add_annotation(
        xref=f"{xb} domain", yref=f"{yb} domain", x=0.99, y=0.5, xanchor="right", yanchor="middle",
        showarrow=False, align="left",
        text="<br>".join(f"{k} = {v:.2f}" for k, v in st.items()),
        bgcolor="rgba(22,25,30,0.9)", bordercolor="#555b66", borderwidth=1, font=dict(size=11, color=JASNY))

    # Podziałka osi Y: co 1 dla godzin i wieku, co 5 (z siatką pomocniczą co 1) dla ocen w punktach
    krok, krok_pom = (5, 1) if jedn == "pkt" else (1, 0.5)
    zakres = st["max"] - st["min"]
    fig.update_xaxes(range=[-0.4, 1.5], showticklabels=False, showgrid=False,
                     minor=dict(showgrid=False), row=i, col=2)
    fig.update_yaxes(title_text=f"[{jedn}]", title_font_size=10, dtick=krok,
                     minor=dict(dtick=krok_pom, showgrid=True, gridcolor=SIATKA_POM, griddash="dot"),
                     range=[st["min"] - zakres * 0.06, st["max"] + zakres * 0.06], row=i, col=2)

# ---------- Gęsta siatka na histogramach: główna + pomocnicza (kropkowana)
fig.update_xaxes(nticks=20, minor=dict(nticks=5, showgrid=True, gridcolor=SIATKA_POM, griddash="dot"), col=1)
fig.update_yaxes(nticks=12, minor=dict(nticks=5, showgrid=True, gridcolor=SIATKA_POM, griddash="dot"), col=1)
fig.update_xaxes(gridcolor=SIATKA, zeroline=False, linecolor="#555b66")
fig.update_yaxes(gridcolor=SIATKA, zeroline=False, linecolor="#555b66")

fig.update_layout(height=340 * len(ilosciowe), width=1150, template="plotly_dark", bargap=0,
                  paper_bgcolor=TLO, plot_bgcolor=TLO, font=dict(color=JASNY),
                  title=dict(text="Rozkłady zmiennych ilościowych", x=0.5),
                  margin=dict(t=80, l=60, r=20, b=40))
fig.show()

# Ile obserwacji leży poza regułą 1,5 x IQR (potencjalne obserwacje odstające; na wykresie ich nie widać, bo wąsy sięgają min/max)
Q1, Q3 = df[ilosciowe].quantile(0.25), df[ilosciowe].quantile(0.75)
IQR = Q3 - Q1
poza = ((df[ilosciowe] < Q1 - 1.5 * IQR) | (df[ilosciowe] > Q3 + 1.5 * IQR)).sum()
pd.DataFrame({"odstające (n)": poza, "odstające [%]": (poza / len(df) * 100).round(2)})

**Wnioski z Części I:**
- **Wiek** ma rozkład jednostajny (13–25 lat) i nie ma obserwacji odstających.
- **Czas w social mediach, czas z AI i aktywność fizyczna** mają rozkłady prawoskośne z wyraźnym słupkiem przy 0
  (ucięty rozkład normalny, ślad generowania danych). Odstające wartości leżą po prawej stronie: pojedyncze osoby z bardzo długim czasem.
- **Sen** ma rozkład zbliżony do normalnego, symetryczny. Odstające wartości są po obu stronach (poniżej ok. 3 h i powyżej ok. 10 h).
- **Oceny zdrowia** są lewoskośne. Odstające wartości to osoby z bardzo niskim wynikiem. Zdrowie fizyczne
  ma dodatkowo „sufit” przy 100 (dużo wyników 97–100).
- Poza wąsami leży łącznie niewiele obserwacji (0–1,2% na zmienną). To naturalne krańce rozkładów, a nie błędy,
  więc zostawiamy je w danych. Sprawdzimy, czy te same obiekty wyróżnią się w PCA, MDS i t-SNE.

---
# Część II: wizualizacja, metody redukcji wymiarów

## 2.0 Przygotowanie danych

- **Zmienne:** 7 zmiennych ilościowych (bez identyfikatora i bez zmiennych jakościowych), wszystkie 16 000 obserwacji.
- **Dane wycentrowane** (odjęta średnia): wejście PCA na macierzy kowariancji.
- **Dane standaryzowane** (odjęta średnia, podzielone przez odchylenie standardowe): wejście PCA na macierzy korelacji.

In [ ]:
from sklearn.preprocessing import StandardScaler

X = df[ilosciowe]                                   # 16 000 x 7
X_c = X - X.mean()                                  # wycentrowane
X_std = pd.DataFrame(StandardScaler().fit_transform(X), columns=ilosciowe, index=X.index)   # standaryzowane

# Krótkie nazwy zmiennych do opisów na wykresach i we wzorach
SKROTY = {"Age": "Age", "Daily_Social_Media_Hours": "SocialMedia", "Daily_AI_Tool_Usage_Hours": "AITools",
          "Sleep_Hours": "Sleep", "Physical_Activity_Hours": "Activity",
          "Mental_Health_Score": "MentalHealth", "Physical_Health_Score": "PhysicalHealth"}

# Kontrola: po standaryzacji każda zmienna ma średnią 0 i odchylenie 1
pd.DataFrame({"x̄ oryg.": X.mean(), "s oryg.": X.std(),
              "x̄ po std.": X_std.mean(), "s po std.": X_std.std(ddof=0)}).round(3)

## 2.1 Macierz kowariancji i macierz korelacji

- **Macierz kowariancji:** na przekątnej wariancje zmiennych, poza nią kowariancje par. Wartości zależą od jednostek.
- **Macierz korelacji:** kowariancje danych standaryzowanych. Na przekątnej same jedynki, poza nią wartości od −1 do 1, bez jednostek.

In [ ]:
def mapa_macierzy(M, tytul, skala):
    """Mapa cieplna macierzy z wartościami w komórkach (czerwone: ujemne, szare: 0, niebieskie: dodatnie)."""
    nazwy = [SKROTY[k] for k in M.columns]
    fig = go.Figure(go.Heatmap(
        z=M.values, x=nazwy, y=nazwy, zmid=0,
        colorscale=[[0, "#ff6b6b"], [0.5, "#2b3038"], [1, "#4c8bd6"]],
        text=M.round(2).values, texttemplate="%{text}", textfont=dict(size=12, color=JASNY),
        hovertemplate="%{y} × %{x}: %{z:.3f}<extra></extra>", colorbar=dict(title=skala)))
    fig.update_yaxes(autorange="reversed")
    fig.update_layout(template="plotly_dark", paper_bgcolor=TLO, plot_bgcolor=TLO, font=dict(color=JASNY),
                      title=dict(text=tytul, x=0.5), width=720, height=600)
    fig.show()

C = X.cov()
R = X.corr()
mapa_macierzy(C, "Macierz kowariancji", "cov")
mapa_macierzy(R, "Macierz korelacji", "r")

# Udział wariancji każdej zmiennej w całkowitej zmienności (ślad macierzy kowariancji)
slad = np.trace(C)
print(f"Całkowita zmienność: ślad macierzy kowariancji = {slad:.2f}, ślad macierzy korelacji = {np.trace(R):.0f}")
pd.DataFrame({"wariancja": np.diag(C), "udział [%]": np.diag(C) / slad * 100},
             index=[SKROTY[k] for k in C.columns]).round(2)

**Wniosek:** w macierzy kowariancji oceny zdrowia (skala 0–100 pkt) mają wariancje 85–94 i odpowiadają razem za ok. **87,5%**
całkowitej zmienności. Wynika to z jednostek, a nie z ważności zmiennych. W macierzy korelacji każda zmienna ma wariancję 1,
więc wszystkie mają równy udział. Najsilniejsze korelacje: SocialMedia z MentalHealth (−0,40), Activity z PhysicalHealth (0,42),
SocialMedia z PhysicalHealth (−0,31) oraz Sleep z MentalHealth (0,31). Wiek nie jest skorelowany z niczym.

### Funkcje do analizy PCA

Te same funkcje wykorzystujemy dla obu macierzy (punkty 2.2 i 2.3).

In [ ]:
from sklearn.decomposition import PCA
from IPython.display import Markdown, display

PC = lambda k: [f"PC{i}" for i in range(1, k + 1)]

def tabela_wariancji(pca):
    """Wartość własna, % zmienności i % skumulowany każdej składowej."""
    proc = pca.explained_variance_ratio_ * 100
    return pd.DataFrame({"wartość własna": pca.explained_variance_, "zmienność [%]": proc,
                         "skumulowana [%]": np.cumsum(proc)}, index=PC(len(proc))).round(2)

def ile_skladowych(pca, progi=(90, 95, 99)):
    """Najmniejsza liczba składowych przenosząca co najmniej dany procent zmienności."""
    skum = np.cumsum(pca.explained_variance_ratio_) * 100
    k = [int(np.argmax(skum >= p - 1e-9)) + 1 for p in progi]   # argmax znajduje pierwsze True
    return pd.DataFrame({"liczba składowych": k, "przenoszona zmienność [%]": [round(skum[n - 1], 2) for n in k]},
                        index=pd.Index(progi, name="próg [%]"))

def wykres_wariancji(pca, tytul, progi=(90, 95, 99)):
    """Słupki: % zmienności składowej; linia: % skumulowany; poziome linie: progi 90/95/99%."""
    proc = pca.explained_variance_ratio_ * 100
    fig = go.Figure()
    fig.add_trace(go.Bar(x=PC(len(proc)), y=proc, name="zmienność składowej", marker_color=KOLOR,
                         text=[f"{p:.1f}%" for p in proc], textposition="outside",
                         hovertemplate="%{x}: %{y:.2f}%<extra></extra>"))
    fig.add_trace(go.Scatter(x=PC(len(proc)), y=np.cumsum(proc), name="skumulowana", mode="lines+markers+text",
                             text=[""] + [f"{p:.1f}%" for p in np.cumsum(proc)[1:]], textposition="top left",   # przy PC1 = słupek
                             line=dict(color="#f0883e", width=2), marker=dict(size=8),
                             hovertemplate="PC1–%{x}: %{y:.2f}%<extra></extra>"))
    for p, kolor in zip(progi, ["#5cc27a", "#e3b341", CZERWONY]):
        fig.add_hline(y=p, line=dict(color=kolor, dash="dash", width=1),
                      annotation_text=f"{p}%", annotation_position="right", annotation_font_color=kolor)
    fig.update_yaxes(range=[0, 112], dtick=10, title_text="% zmienności",
                     minor=dict(dtick=5, showgrid=True, gridcolor="#262a30", griddash="dot"))
    fig.update_layout(template="plotly_dark", paper_bgcolor=TLO, plot_bgcolor=TLO, font=dict(color=JASNY),
                      title=dict(text=tytul, x=0.5), height=460, width=950,
                      legend=dict(orientation="h", y=-0.15, x=0.5, xanchor="center"))
    fig.show()

def rownania(pca, dane, symbol, k=3):
    """Postać matematyczna k pierwszych składowych: PC = suma (ładunek x zmienna)."""
    nazwy = [SKROTY[c] for c in dane.columns]
    for i in range(k):
        skladniki = " ".join(f"{a:+.3f}\\,{symbol}_{{{n}}}" for a, n in zip(pca.components_[i], nazwy))
        display(Markdown(f"$$PC_{i+1} = {skladniki}$$"))

def wykres_2d_3d(pca, dane, tytul, koloruj=("Mental_Health_Score", "Daily_Social_Media_Hours")):
    """Obserwacje w układzie PC1 × PC2 (2D, osobny wykres dla każdej zmiennej z `koloruj`)
    oraz PC1 × PC2 × PC3 (3D, kolor według pierwszej zmiennej). Najechanie pokazuje Student_ID."""
    Z = pca.transform(dane)[:, :3]
    proc = pca.explained_variance_ratio_ * 100
    os_ = [f"PC{i+1} ({proc[i]:.1f}%)" for i in range(3)]
    SKALA = "Plasma"   # skala sekwencyjna: ciemny fiolet = niska wartość, żółty = wysoka

    fig = make_subplots(rows=1, cols=len(koloruj), horizontal_spacing=0.14,
                        subplot_titles=[f"kolor: {SKROTY[k]}" for k in koloruj])
    for j, kol in enumerate(koloruj, start=1):
        fig.add_trace(go.Scattergl(
            x=Z[:, 0], y=Z[:, 1], mode="markers", showlegend=False,
            customdata=np.c_[df["Student_ID"], df[kol]],
            marker=dict(size=3, opacity=0.7, color=df[kol], colorscale=SKALA,
                        colorbar=dict(title=SKROTY[kol], len=0.85, thickness=12,
                                      x=1.01 if j == len(koloruj) else j / len(koloruj) - 0.06)),   # pasek koloru obok swojego wykresu
            hovertemplate=f"%{{customdata[0]}}<br>{SKROTY[kol]} = %{{customdata[1]}}"
                          "<br>PC1 = %{x:.2f}<br>PC2 = %{y:.2f}<extra></extra>"), row=1, col=j)
        fig.update_xaxes(title_text=os_[0], row=1, col=j)
        fig.update_yaxes(title_text=os_[1], row=1, col=j)
    fig.update_xaxes(zeroline=True, zerolinecolor="#777777")
    fig.update_yaxes(zeroline=True, zerolinecolor="#777777")
    fig.update_layout(template="plotly_dark", paper_bgcolor=TLO, plot_bgcolor=TLO, font=dict(color=JASNY),
                      title=dict(text=f"{tytul}: PC1 × PC2", x=0.5), height=520, width=1200)
    fig.show()

    kol = koloruj[0]
    fig = go.Figure(go.Scatter3d(
        x=Z[:, 0], y=Z[:, 1], z=Z[:, 2], mode="markers", customdata=np.c_[df["Student_ID"], df[kol]],
        marker=dict(size=2, opacity=0.7, color=df[kol], colorscale=SKALA,
                    colorbar=dict(title=SKROTY[kol], len=0.7, thickness=12)),
        hovertemplate=f"%{{customdata[0]}}<br>{SKROTY[kol]} = %{{customdata[1]}}"
                      "<br>PC1 = %{x:.2f}<br>PC2 = %{y:.2f}<br>PC3 = %{z:.2f}<extra></extra>"))
    fig.update_layout(template="plotly_dark", paper_bgcolor=TLO, font=dict(color=JASNY),
                      title=dict(text=f"{tytul}: PC1 × PC2 × PC3 (kolor: {SKROTY[kol]})", x=0.5),
                      height=650, width=880,
                      scene=dict(xaxis_title=os_[0], yaxis_title=os_[1], zaxis_title=os_[2], bgcolor=TLO))
    fig.show()


## 2.2 PCA na macierzy kowariancji

### Udział składowych w zmienności

In [ ]:
pca_cov = PCA().fit(X_c)          # dane wycentrowane, czyli PCA na macierzy kowariancji
tabela_wariancji(pca_cov)

### Ile składowych przenosi 90%, 95% i 99% zmienności?

In [ ]:
wykres_wariancji(pca_cov, "PCA (macierz kowariancji): zmienność przenoszona przez składowe")
ile_skladowych(pca_cov)

### Wykres 2D (PC1, PC2) i 3D (PC1, PC2, PC3)

In [ ]:
wykres_2d_3d(pca_cov, X_c, "PCA (macierz kowariancji)")

### Postać matematyczna trzech pierwszych składowych

$\tilde{x}$ oznacza zmienną wycentrowaną ($x - \bar{x}$), a współczynniki to ładunki, czyli wagi zmiennych w składowej.

In [ ]:
rownania(pca_cov, X_c, symbol="\\tilde{x}")

**Odpowiedzi (macierz kowariancji):**
- **90%** zmienności przenoszą **3** składowe (95,34%), **95%** również **3** składowe, **99%** dopiero **6** składowych (99,60%).
- Pierwsze dwie składowe przenoszą **88,42%** zmienności, pierwsze trzy **95,34%**.
- PC1 i PC2 to praktycznie tylko MentalHealth i PhysicalHealth (ładunki 0,64–0,76): PC1 to ich suma (ogólne zdrowie),
  PC2 to ich różnica. PC3 to sam wiek (ładunek 1,000). Zmienne mierzone w godzinach mają ładunki bliskie zera, bo ich
  wariancje są małe w porównaniu z ocenami 0–100 pkt. Wynik jest zdominowany przez jednostki zmiennych.
- **Kolory na mapie 2D:** MentalHealth rośnie po przekątnej w prawo i w górę (zgodnie z ładunkami PC1 i PC2), a SocialMedia nie układa się w wyraźny gradient, bo w tej analizie prawie nie wpływa na składowe.

## 2.3 PCA na macierzy korelacji

### Udział składowych w zmienności

In [ ]:
pca_kor = PCA().fit(X_std)        # dane standaryzowane, czyli PCA na macierzy korelacji
tabela_wariancji(pca_kor)

### Ile składowych przenosi 90%, 95% i 99% zmienności?

In [ ]:
wykres_wariancji(pca_kor, "PCA (macierz korelacji): zmienność przenoszona przez składowe")
ile_skladowych(pca_kor)

### Wykres 2D (PC1, PC2) i 3D (PC1, PC2, PC3)

In [ ]:
wykres_2d_3d(pca_kor, X_std, "PCA (macierz korelacji)")

### Postać matematyczna trzech pierwszych składowych

$z$ oznacza zmienną standaryzowaną ($(x - \bar{x}) / s$).

In [ ]:
rownania(pca_kor, X_std, symbol="z")

**Odpowiedzi (macierz korelacji):**
- **90%** zmienności przenosi **6** składowych (93,26%), a **95%** i **99%** dopiero wszystkie **7** składowych (100%).
- Pierwsze dwie składowe przenoszą **46,25%** zmienności, pierwsze trzy **60,53%**.
- PC1 to oś „zdrowego stylu życia”: z jednej strony dużo snu, aktywności oraz wysokie oceny zdrowia (ładunki od +0,37 do +0,49),
  z drugiej dużo czasu w mediach społecznościowych (−0,47). PC2 łączy głównie korzystanie z AI i aktywność (+0,62, +0,55)
  przy krótszym śnie (−0,42). PC3 to znowu sam wiek (0,987).
- **Kolory na mapie 2D** potwierdzają znaczenie PC1: przesuwając się w prawo, MentalHealth rośnie (fiolet → żółty), a SocialMedia maleje (żółty → fiolet). Prawa strona mapy to studenci zdrowsi i rzadziej korzystający z mediów społecznościowych.

**Porównanie obu macierzy:** po standaryzacji wszystkie zmienne mają równy wpływ, więc składowe opisują rzeczywiste
zależności między zmiennymi, a nie różnice skal. Kosztem tego jest dużo wolniejsze narastanie wariancji: zmienne są ze sobą
słabo skorelowane (|r| ≤ 0,42), więc nie da się ich dobrze streścić kilkoma składowymi i do 90% potrzeba aż 6 z 7.

## 2.4 MDS (skalowanie wielowymiarowe)

In [69]:
# TODO: MDS metryczny, mapa 2D
# podpowiedź: sklearn.manifold.MDS(n_components=2, metric=True, random_state=42)

In [70]:
# TODO: MDS niemetryczny, mapa 2D (metric=False)

In [71]:
# TODO: wykres wartości Stress od liczby wymiarów (np. 1–6)
# podpowiedź: pętla po n_components, atrybut .stress_

**Wnioski z MDS:** _uzupełnij (różnice między metodą metryczną i niemetryczną, ile wymiarów wystarcza)._

## 2.5 t-SNE

In [72]:
# TODO: t-SNE, mapa 2D
# podpowiedź: sklearn.manifold.TSNE(n_components=2, perplexity=30, random_state=42)

## 2.6 Obserwacje odstające

In [73]:
# TODO: wskaż obserwacje odstające (np. odległość od środka w przestrzeni PCA albo z-score)
# i zaznacz te same punkty na mapach PCA, MDS i t-SNE

**Czy na mapach da się zlokalizować obserwacje odstające?** _uzupełnij_

**Czy są to te same obiekty co w PCA?** _uzupełnij_

---
# Podsumowanie

_Uzupełnij._